In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/06_genai/05_sql_executor.py

# Phase 13 — Visualization

This notebook converts validated SQL results into appropriate
business visualizations.

The visualization layer:

1. Receives actual SQL results.
2. Determines the appropriate chart type.
3. Converts the result into chart-ready records.
4. Never invents analytical values.
5. Uses the actual Spark SQL result as the source of truth.

Supported charts:

- bar
- line
- pie
- scatter
- table

In [0]:
import json
from typing import Dict, Any, List

from pyspark.sql import functions as F

In [0]:
SUPPORTED_CHARTS = {
    "bar",
    "line",
    "pie",
    "scatter",
    "table"
}

MAX_CHART_ROWS = 50

print("Supported chart types:")
for chart in sorted(SUPPORTED_CHARTS):
    print("-", chart)

In [0]:
def get_result_columns(result: Dict[str, Any]) -> List[str]:
    """
    Return columns from a successful SQL execution result.
    """

    if not result.get("success"):
        return []

    dataframe = result.get("dataframe")

    if dataframe is None:
        return []

    return dataframe.columns

In [0]:
def get_numeric_columns(dataframe):
    """
    Return numeric columns from a Spark DataFrame.
    """

    numeric_types = {
        "byte",
        "short",
        "int",
        "long",
        "float",
        "double",
        "decimal"
    }

    numeric_columns = []

    for field in dataframe.schema.fields:
        data_type = field.dataType.simpleString().lower()

        if any(
            data_type.startswith(numeric_type)
            for numeric_type in numeric_types
        ):
            numeric_columns.append(field.name)

    return numeric_columns

In [0]:
def get_categorical_columns(dataframe):
    """
    Return string/categorical columns.
    """

    categorical_columns = []

    for field in dataframe.schema.fields:

        data_type = field.dataType.simpleString().lower()

        if data_type == "string":
            categorical_columns.append(field.name)

    return categorical_columns

In [0]:
def detect_chart_type(
    dataframe,
    question: str = ""
) -> str:

    question_lower = question.lower()

    columns = dataframe.columns

    numeric_columns = get_numeric_columns(dataframe)
    categorical_columns = get_categorical_columns(dataframe)

    # Empty result
    if len(columns) == 0:
        return "table"

    # Scatter:
    # Two or more numeric fields often indicate a relationship.
    if len(numeric_columns) >= 2:
        relationship_keywords = [
            "relationship",
            "correlation",
            "versus",
            "vs",
            "against",
            "impact",
            "relationship between"
        ]

        if any(
            keyword in question_lower
            for keyword in relationship_keywords
        ):
            return "scatter"

    # Trend / time series
    trend_keywords = [
        "trend",
        "over time",
        "monthly",
        "month over month",
        "year over year",
        "daily",
        "weekly",
        "quarterly",
        "change over time"
    ]

    if any(
        keyword in question_lower
        for keyword in trend_keywords
    ):
        return "line"

    # Ranking / comparison
    ranking_keywords = [
        "top",
        "highest",
        "lowest",
        "best",
        "worst",
        "compare",
        "comparison",
        "by region",
        "by category",
        "by customer"
    ]

    if any(
        keyword in question_lower
        for keyword in ranking_keywords
    ):
        return "bar"

    # Part-to-whole
    share_keywords = [
        "share",
        "percentage of total",
        "proportion",
        "composition",
        "distribution"
    ]

    if any(
        keyword in question_lower
        for keyword in share_keywords
    ):
        if len(categorical_columns) == 1 and len(numeric_columns) == 1:
            return "pie"

    # Default analytical visualization
    if len(categorical_columns) >= 1 and len(numeric_columns) >= 1:
        return "bar"

    return "table"

In [0]:
def recommend_visualization(
    dataframe,
    question: str
) -> Dict[str, Any]:

    chart_type = detect_chart_type(
        dataframe,
        question
    )

    return {
        "chart_type": chart_type,
        "question": question,
        "columns": dataframe.columns,
        "numeric_columns": get_numeric_columns(dataframe),
        "categorical_columns": get_categorical_columns(dataframe)
    }

In [0]:
def dataframe_to_records(
    dataframe,
    max_rows: int = MAX_CHART_ROWS
) -> List[Dict[str, Any]]:
    """
    Convert Spark DataFrame into JSON-compatible records.
    """

    rows = dataframe.limit(max_rows).collect()

    records = []

    for row in rows:
        record = row.asDict(recursive=True)

        cleaned_record = {}

        for key, value in record.items():

            if value is None:
                cleaned_record[key] = None

            elif hasattr(value, "isoformat"):
                cleaned_record[key] = value.isoformat()

            else:
                cleaned_record[key] = value

        records.append(cleaned_record)

    return records

In [0]:
def build_visualization_spec(
    result: Dict[str, Any],
    question: str
) -> Dict[str, Any]:

    if not result.get("success"):
        return {
            "success": False,
            "chart_type": "table",
            "message": "Visualization unavailable because SQL execution failed.",
            "data": []
        }

    dataframe = result.get("dataframe")

    if dataframe is None:
        return {
            "success": False,
            "chart_type": "table",
            "message": "No result DataFrame was returned.",
            "data": []
        }

    recommendation = recommend_visualization(
        dataframe,
        question
    )

    chart_type = recommendation["chart_type"]

    records = dataframe_to_records(
        dataframe
    )

    return {
        "success": True,
        "chart_type": chart_type,
        "question": question,
        "columns": dataframe.columns,
        "numeric_columns": recommendation["numeric_columns"],
        "categorical_columns": recommendation["categorical_columns"],
        "row_count": len(records),
        "data": records
    }

In [0]:
question = "Which region generated the highest revenue?"

sql = """
SELECT
    region,
    SUM(total_revenue) AS total_revenue
FROM genai_copilot.gold.region_sales
GROUP BY region
ORDER BY total_revenue DESC
LIMIT 10
"""

In [0]:
result = execute_sql(sql)

print(
    json.dumps(
        {
            "success": result["success"],
            "row_count": result["row_count"],
            "execution_time_ms": result["execution_time_ms"],
            "error": result["error"]
        },
        indent=2,
        default=str
    )
)

In [0]:
visualization = build_visualization_spec(
    result=result,
    question=question
)

print(
    json.dumps(
        visualization,
        indent=2,
        default=str
    )
)

In [0]:
def get_chart_fields(dataframe):
    """
    Determine suitable x and y fields for common analytical results.
    """

    categorical_columns = get_categorical_columns(dataframe)
    numeric_columns = get_numeric_columns(dataframe)

    x_column = (
        categorical_columns[0]
        if categorical_columns
        else dataframe.columns[0]
    )

    y_column = (
        numeric_columns[0]
        if numeric_columns
        else None
    )

    return {
        "x_column": x_column,
        "y_column": y_column
    }

In [0]:
fields = get_chart_fields(
    result["dataframe"]
)

print(fields)

In [0]:
def build_chart_config(
    result: Dict[str, Any],
    question: str
) -> Dict[str, Any]:

    if not result.get("success"):
        return {
            "chart_type": "table",
            "message": "No chart available."
        }

    dataframe = result["dataframe"]

    recommendation = recommend_visualization(
        dataframe,
        question
    )

    fields = get_chart_fields(
        dataframe
    )

    return {
        "chart_type": recommendation["chart_type"],
        "x_column": fields["x_column"],
        "y_column": fields["y_column"],
        "data": dataframe_to_records(dataframe),
        "question": question
    }

In [0]:
chart_config = build_chart_config(
    result,
    question
)

print(
    json.dumps(
        chart_config,
        indent=2,
        default=str
    )
)

### Test

In [0]:
question = "Which region generated the highest revenue?"

In [0]:
question = "What is the monthly revenue trend?"

In [0]:
print("Visualization test")
print("=" * 50)

print("Question:", question)
print("SQL success:", result["success"])
print("Rows:", result["row_count"])
print("Chart:", visualization["chart_type"])
print("X column:", chart_config["x_column"])
print("Y column:", chart_config["y_column"])


What Phase 13 demonstrates

You now have:

✅ PySpark result handling
✅ Automatic chart selection
✅ Actual SQL result → visualization
✅ Bar chart support
✅ Line chart support
✅ Pie chart support
✅ Scatter chart support
✅ Table fallback
✅ No fabricated numbers
✅ Controlled maximum chart rows

And the overall GenAI pipeline is becoming:

01 Schema Profiler
        ↓
02 Question Classifier
        ↓
03 SQL Generator
        ↓
04 SQL Validator
        ↓
05 SQL Executor
        ↓
06 Answer Generator
        ↓
07 Visualization


